# 数据类型与变量

> **开始前请注意：在线输入限制**
> 当前在线环境中的 `scanf`、`getchar`、`fgets(..., stdin)` 无法交互读取键盘输入。在线实验请修改变量的初值，或使用 `sscanf` 从字符串读取；键盘输入练习请在本地 GCC / Clang 中运行完整 C 程序。

**自测说明**：本章应用题的未完成模板会显示 FAIL；填写答案区域后应全部 PASS。这是作答反馈，不是内核故障。

**本章目标**：用 sizeof 和头文件宏观察当前平台；解释整数除法、取模和类型转换；为输出选择正确的格式。

**学习方法**：先预测输出，再运行验证；每次只修改一个条件，最后用自己的话解释变化。修改函数或类型定义后，请重启内核并从头运行。

在 C 语言中，**变量** 就像一个容器，用来存储数据。不同类型的数据需要不同类型的容器，这就是 **数据类型** 的作用。

本节将学习：
- 常见数据类型
- 如何声明和使用变量
- 输入输出变量
- 类型转换

## 先理解变量：初始化与赋值

`int score = 58;` 声明变量并初始化；后面的 `score = 60;` 给已有变量赋新值。`=` 改变左侧对象的值，`==` 比较两边的值。

`score += 2` 在这里等价于 `score = score + 2`。`++score` 与 `score++` 单独作为语句时都加一；作为表达式时，前者取得更新后的值，后者取得更新前的值。初学时将自增单独写一行，避免在一个表达式里多次修改同一变量。

变量名区分大小写；这里用字母开头、字母数字和下划线组成的名字，不使用关键字。局部变量先初始化再读取。`const int pass_line = 60;` 表示不能再通过该名字给对象赋值。

**先画状态表**：每条语句执行后 score 是多少？最后两次比较各得到什么？

In [ ]:
#include <stdio.h>
{
    int score = 58;
    const int pass_line = 60;
    printf("初值=%d\n", score);
    score = 59;
    score += 2;
    ++score;
    printf("现值=%d 等于及格线=%d 达到及格线=%d\n",
           score, score == pass_line, score >= pass_line);
}

## 1. 常见数据类型

- 整数类型：`short`、`int`、`long`、`long long`，均有带符号和无符号形式。
- `char`、`signed char`、`unsigned char` 是三种不同的类型；普通 `char` 的有无符号由实现决定。
- `char` 既可存储字符编码，也可参与整数运算。C 标准不要求所有实现使用 ASCII；本课程浏览器环境使用 ASCII 兼容编码。
- `float`、`double`、`long double` 表示浮点数。小数常量如 `3.14` 默认是 `double`；`3.14f` 是 `float`。

### 存储大小：标准保证与平台实测

C17 不规定 `int`、`long` 必须占多少个字节，也不能仅根据操作系统“64 位”判断。

| 标准保证 | 如何检查当前实现 |
| --- | --- |
| `sizeof(char) == 1`；一个 C 字节至少有 8 位 | `CHAR_BIT` |
| `sizeof(short) <= sizeof(int) <= sizeof(long) <= sizeof(long long)` | `sizeof` |
| `int` 至少能表示 -32767 到 32767 | `INT_MIN`、`INT_MAX` |
| `long` 至少能表示 -2147483647 到 2147483647 | `LONG_MIN`、`LONG_MAX` |

使用 `<limits.h>` 的宏查看整数范围。`sizeof` 的结果是 `size_t`，输出格式为 `%zu`。不要把某一次实验的数字当成所有平台的规则。

### 浮点型

常见平台的 `float` 为 32 位、`double` 为 64 位，但应使用 `sizeof` 与 `<float.h>` 观察当前实现。

`FLT_MIN` / `DBL_MIN` 是**最小正规格化正值**，不是最负的数，也不一定是最小可表示正值。`FLT_MAX` 是最大有限正值；`FLT_DIG` 描述十进制有效数字能力。浮点数不能精确表示所有十进制小数。

In [ ]:
#include <stdio.h>
#include <float.h>
{
    printf("float 占用字节数 : %zu \n", sizeof(float));
    printf("float 最小正规格化正值: %E\n", FLT_MIN );
    printf("float 最大值: %E\n", FLT_MAX );
    printf("精度值: %d\n", FLT_DIG );

    printf("double 占用字节数 : %zu \n", sizeof(double));
    printf("double 最小正规格化正值: %E\n", DBL_MIN );
    printf("double 最大值: %E\n", DBL_MAX );
    printf("精度值: %d\n", DBL_DIG );
}

## 2. 声明变量

语法：
```c
类型 变量名 = 初始值;
```

示例：

In [ ]:
#include <stdio.h>
{
    unsigned int age = 18;
    float pi = 3.14f;
    char grade = 'A';
    
    printf("年龄: %u\n", age);
    printf("圆周率: %f\n", pi);
    printf("成绩等级: %c\n", grade);
}

为了得到某个类型或某个变量在特定平台上的准确大小，您可以使用 sizeof 运算符。表达式 sizeof(type) 得到对象或类型的存储字节大小。

示例如下：

In [ ]:
{
    printf("int 存储大小 : %zu \n", sizeof(int));

    unsigned short x=10;
    printf("x 存储大小 : %zu \n", sizeof(x));

    long y=10;
    printf("y 存储大小 : %zu \n", sizeof(y));

    float z=10.;
    printf("z 存储大小 : %zu \n", sizeof(z));
}

## 3. 变量的输入

本地 C 程序使用 `scanf` 从标准输入读取数据，并检查返回值。

```c
#include <stdio.h>

int main(void) {
    int number;
    printf("请输入一个整数: ");
    if (scanf("%d", &number) != 1) {
        fprintf(stderr, "输入无效，需要一个整数。\n");
        return 1;
    }
    printf("你输入的是: %d\n", number);
    return 0;
}
```

当前浏览器版 C17 内核的 `scanf` 返回 EOF，不会弹出输入框。下面使用明确给定的数据进行在线实验；这不是键盘输入。

In [ ]:
{
    int number = 10; // 修改这里的值，再运行本单元
    printf("实验数据: %d\n", number);
}

📌 注意：
- `scanf("%d", &number)` 中的 `&` 表示变量地址，返回 1 表示成功读取一个整数。
- `sscanf` 从字符串读取数据，适合在当前在线环境中练习格式化读取。
- 修改下面的字符串为 `"abc"`，观察读取失败时的分支。
- `getchar`、`fgets` 在当前浏览器内核中也无法交互读取标准输入；键盘输入请用本地 C 编译器练习。

In [ ]:
{
    const char input[] = "42";
    int number;
    if (sscanf(input, "%d", &number) == 1) {
        printf("从字符串读取: %d\n", number);
    } else {
        printf("读取失败：字符串中没有有效整数。\n");
    }
}

## 4. 多个变量

你可以同时声明多个同类型变量。

In [ ]:
{
    int x = 3, y = 4;
    int sum = x + y;
    printf("x = %d, y = %d, 和 = %d\n", x, y, sum);
}

## 5. 类型转换

混合运算会按整数提升、通常算术转换等规则确定运算类型，不是简单地“把小范围提升到大范围”。例如 `int` 与 `double` 相加时会转为 `double`；有符号数与无符号数混用则需格外小心。

赋值转换也可能丢失信息：在可表示范围内，浮点数转为整数会舍弃小数部分（向零截断）。本课程不运行超出目标范围的转换。

In [ ]:
{
    int a = 5;
    double b = 2.0;
    double result = a + b;
    printf("混合相加: %.1f\n", result);
    printf("向零截断: %d\n", (int)-3.8);
}

### 强制转换（显式）
使用 `(类型)` 进行强制转换：

In [ ]:
{
    int a = 7, b = 2;
    double result = (double)a / b;  //变量a的值被强制转换为double类型用于计算，但需注意变量a本身的类型并不会改变
    printf("7 / 2 = %f\n", result);
}

## 实验 1：测量当前平台

**预测**：浏览器中的 `long` 一定是 8 字节吗？**运行**后记录结果，再用本地编译器比较。

**解释**：C 字节数、整数范围和指针宽度由目标平台决定；浏览器的 WebAssembly 平台与宿主操作系统不必相同。

In [ ]:
#include <limits.h>
{
    printf("CHAR_BIT=%d\n", CHAR_BIT);
    printf("int: %zu bytes, %d .. %d\n", sizeof(int), INT_MIN, INT_MAX);
    printf("long: %zu bytes, %ld .. %ld\n", sizeof(long), LONG_MIN, LONG_MAX);
}

## 实验 2：整数除法、取模与转换时机

**预测**下面四个结果，运行后解释为什么 `(double)(a / b)` 仍没有保留原来的小数部分。将 `a` 改成 -7，观察向零截断以及余数的符号。除数必须非零。

In [ ]:
{
    int a = 7, b = 2;
    printf("整数商=%d, 余数=%d\n", a / b, a % b);
    printf("先除后转=%.1f, 先转后除=%.1f\n", (double)(a / b), (double)a / b);
}

## 实验 3：逻辑、优先级与混合符号

关系和逻辑运算的结果为 0 或 1。`=` 是赋值，`==` 是相等比较。用括号明确希望先算的部分。

**预测**：`-1 < 1U` 是否为真？运行后解释：本例比较前，`-1` 转为 `unsigned int`，因此不能按两个数学整数直接比较。不要把此结果推广为所有不同整数类型的组合。

In [ ]:
{
    int negative = -1;
    unsigned int positive = 1U;
    printf("优先级: %d %d\n", 2 + 3 * 4, (2 + 3) * 4);
    printf("比较: %d %d\n", negative < 1, negative < positive);
    printf("逻辑: %d %d\n", 85 >= 60 && 85 <= 100, !(85 >= 60));
}

## 贯穿案例 2：计算一位学生的成绩

平时成绩 85，考试成绩 90，权重分别为 60% 和 40%。**先算出预期值 87.00**，再运行。将平时成绩改为 0、考试成绩改为 100，结果应为 40.00。权重之和应为 1。

In [ ]:
{
    double daily = 85.0, exam = 90.0;
    double final_score = daily * 0.6 + exam * 0.4;
    printf("综合成绩: %.2f\n", final_score);
}

## 数值观察：显示精度不等于计算精度

在当前内核观察 `0.1 + 0.2` 的高精度输出，再改成保留两位小数。输出看起来一样，不代表内部表示完全一样；浮点数不能精确表示所有十进制小数。

不要把某个固定容差当成所有问题的答案。涉及分数阈值时先明确规则；涉及金额时，可在范围允许的条件下用整数表示“分”。本实验只观察误差，不要求跨平台最后一位输出一致。

In [ ]:
{
    double sum = 0.1 + 0.2;
    printf("高精度: %.17g\n两位小数: %.2f\n", sum, sum);
    printf("与0.3的差: %.17g\n", sum - 0.3);
}

### 读取成功与完整输入合法是两回事

`sscanf` 返回成功完成赋值的转换项数。把前面的 input 改成 `"42abc"`，`%d` 仍能转换出 42；这不说明整个字符串都是整数。当前练习只接受题目约定范围内的数字，不把 `%d` 当成任意长数字文本的安全范围校验器。

## 分层练习

### 1. 读程序
预测 `7 / 2`、`7 % 2`、`7 / 2.0`；然后解释 `double x = 7 / 2;` 的值。

<details><summary>提示：先自己尝试</summary>

先看参与除法的操作数类型，再看赋值目标。

</details>

<details><summary>参考思路与自查</summary>

分别为 3、1、3.5；`x` 是 3.0，因为整数除法已经完成。

</details>

### 2. 改错
`unsigned int count = 3; printf("%d", count);` 和 `printf("%d", sizeof(int));` 应怎样修改？

<details><summary>提示：先自己尝试</summary>

格式说明符要与实际参数类型一致。

</details>

<details><summary>参考思路与自查</summary>

分别改为 `%u` 与 `%zu`；不能只因当前数值较小就忽略类型。

</details>

### 3. 编程
将总秒数 125 转换为“2 分 5 秒”；再检查 0、59、60。

<details><summary>提示：先自己尝试</summary>

商表示分钟，余数表示剩余秒数。

</details>

<details><summary>参考思路与自查</summary>

使用 `/ 60` 与 `% 60`；三个边界样例应为 0:0、0:59、1:0。

</details>

### 4. 应用
两次成绩为 59 和 60，输出平均值 59.50；再试 0 和 100。

<details><summary>提示：先自己尝试</summary>

在除法发生前引入浮点类型。

</details>

<details><summary>参考思路与自查</summary>

`(a + b) / 2.0`，本题成绩限定在 0..100；第二组为 50.00。

</details>

### 应用练习：先拆分一个时长

将总秒数 input 分解为 hours、minutes、seconds。先只处理3661秒，运行后应显示 `1:1:1`。只填写标记间的赋值语句，保留输入不变。

本题计算时长，86400秒应是24小时；分钟与秒都在0..59。使用 long 容纳0..86400的输入，输出格式为 %ld。

<details><summary>一级提示</summary>

先求完整小时，再处理去掉完整小时后的剩余秒数。

</details>

<details><summary>二级提示</summary>

整数除法取得完整单位，余数用于继续拆分。

</details>

In [ ]:
{
    const long input = 3661;
    long hours = -1, minutes = -1, seconds = -1;
    /* BEGIN ANSWER */
    // 在这里填写三条赋值语句。
    /* END ANSWER */
    printf("%ld:%ld:%ld\n", hours, minutes, seconds);
}

### 完成后再验收：秒数拆分的7组输入

先确认上面的3661秒得到1:1:1，再手动改为59、60、3600观察边界。然后展开下方默认折叠的代码单元，将已完成的答案复制到同名标记区域，运行全部用例。

| 输入 | 期望 时:分:秒 |
| --- | --- |
| 0 | 0:0:0 |
| 59 | 0:0:59 |
| 60 | 0:1:0 |
| 3599 | 0:59:59 |
| 3600 | 1:0:0 |
| 3661 | 1:1:1 |
| 86400 | 24:0:0 |

用例表、循环与反馈由课程提供，稍后学习数组时再阅读；本题只要求理解自己的三条赋值语句。初始模板出现 FAIL 表示答案尚未复制，完成后应为7/7 PASS。解释为什么分钟不能直接写 input / 60。

点击折叠单元的省略号展开代码。添加测试时，inputs 与 expected 必须同时增加对应的一行；数量不一致会在编译时明确报错，不会继续读取用例。

In [ ]:
{
    const long inputs[] = {0, 59, 60, 3599, 3600, 3661, 86400};
    const long expected[][3] = {
        {0,0,0}, {0,0,59}, {0,1,0}, {0,59,59}, {1,0,0}, {1,1,1}, {24,0,0}
    };
    _Static_assert(sizeof inputs / sizeof inputs[0] == sizeof expected / sizeof expected[0],
                   "输入表与期望表长度不一致：请同时增删对应行");
    const size_t case_count = sizeof inputs / sizeof inputs[0];
    size_t passed = 0;
    for (size_t item = 0; item < case_count; ++item) {
        const long input = inputs[item];
        long hours = -1, minutes = -1, seconds = -1;
        /* BEGIN ANSWER */
        // TODO: 计算 hours、minutes、seconds。
        /* END ANSWER */
        int ok = hours == expected[item][0] && minutes == expected[item][1] && seconds == expected[item][2];
        if (ok) { ++passed; }
        printf("%s 输入=%ld: 实际=%ld:%ld:%ld 期望=%ld:%ld:%ld\n",
               ok ? "PASS" : "FAIL", input, hours, minutes, seconds,
               expected[item][0], expected[item][1], expected[item][2]);
    }
    printf("秒数拆分: %zu/%zu PASS\n", passed, case_count);
}

<details><summary>参考答案：完成作答后再展开</summary>

单例与批量作答使用同一段答案。只替换 BEGIN ANSWER 与 END ANSWER 之间的区域，然后重新运行同一个单元。保留验收代码与期望值。

```c
hours = input / 3600;
minutes = input % 3600 / 60;
seconds = input % 60;
```

</details>

## 本地实践：完整 C17 程序

将下面代码保存为 `chapter02.c`，执行 `cc -std=c17 -Wall -Wextra -Wpedantic chapter02.c -o chapter02`，再运行 `./chapter02`（Windows 使用 `chapter02.exe`）。此代码展示完整程序结构，不作为 Notebook 单元执行。

```c
#include <stdio.h>

int main(void) {
    double daily = 85.0, exam = 90.0;
    printf("综合成绩: %.2f\n", daily * 0.6 + exam * 0.4);
    return 0;
}
```

默认样例的标准输出：

```text
综合成绩: 87.00
```

## 小结

先确定操作数类型，再判断运算和转换；用 `%zu` 打印 `sizeof`。平台大小要实测，浮点精度与数值范围是不同概念。